In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision.datasets import CIFAR10

In [2]:
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

trainset = CIFAR10(root="./data", train=True, download=True, transform=transform)
testset = CIFAR10(root="./data", train=False, download=True, transform=transform)

100%|██████████| 170M/170M [00:05<00:00, 29.0MB/s]


In [3]:
trainloader = DataLoader(trainset, batch_size=64, shuffle=True)
testloader = DataLoader(testset, batch_size=64)

In [4]:
class CNN(nn.Module):
  def __init__(self):
    super(CNN, self).__init__()

    self.conv_layers = nn.Sequential(
        nn.Conv2d(3, 32, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2),

        nn.Conv2d(32, 64, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2),

        nn.Conv2d(64, 128, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2),
    )

    self.fc_layers = nn.Sequential(
        nn.Linear(128 * 4 * 4, 256),
        nn.ReLU(),

        nn.Linear(256, 10)
    )

  def forward(self, x):
    x = self.conv_layers(x)
    x = x.view(x.size(0), -1)  #flattening
    x = self.fc_layers(x)
    return x

In [5]:
model = CNN()

In [6]:
criterian = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [7]:
epochs = 10

for epoch in range(epochs):
  epoch_training_loss = 0.0

  for images, labels in trainloader:
    optimizer.zero_grad()

    output = model.forward(images)
    loss = criterian(output, labels)

    loss.backward()
    optimizer.step()

    epoch_training_loss += loss.item()

  print(f"Epoch {epoch + 1}/{epochs}, Training Loss: {epoch_training_loss / len(trainloader)}")


Epoch 1/10, Training Loss: 1.4023589380561847
Epoch 2/10, Training Loss: 0.9624112220981237
Epoch 3/10, Training Loss: 0.7695919351123482
Epoch 4/10, Training Loss: 0.6370340645923029
Epoch 5/10, Training Loss: 0.536651258692717
Epoch 6/10, Training Loss: 0.44292517334146575
Epoch 7/10, Training Loss: 0.35205911823055325
Epoch 8/10, Training Loss: 0.27713824959133593
Epoch 9/10, Training Loss: 0.2136282938939836
Epoch 10/10, Training Loss: 0.171081798255939


In [9]:
correct_labels = 0
total_labels = 0

model.eval()

with torch.no_grad():
  for images, labels in testloader:
    output = model.forward(images)
    __, predicted = torch.max(output, 1)

    correct_labels += (predicted == labels).sum().item()
    total_labels += labels.size(0)

accuracy = correct_labels / total_labels
print(f"Test Accuracy: {accuracy * 100}%")

Test Accuracy: 75.08%
